# Hot-installing vllm#62240 into a running DeepSeek-V4.1-Flash engine

This notebook walks through PR #8 step by step: what the upstream change is, why the obvious way
to hot-patch it silently does nothing, how the live lab installs it instead, and what happened
when we ran it on a warm 4-GPU engine.

**How to read it.** Cells marked *CPU* were executed when this notebook was built; you can rerun
them on any machine with Python 3. Cells marked *GPU, recorded* show the code we ran on
4× GB200 and its real output from that run; they need the engine and are not re-executed here.

## 1. The change we want to hot-install

vllm-project/vllm#62240 changes one function, `sparse_attn_indexer`: when the backend is
`deep_select`, it ranks the prefill indexer's bf16 logits with DeepSelect instead of upcasting to
fp32 and using the per-row top-k. The blog reports +15.6% prefill throughput at 32K new tokens.

```diff
- logits = as_fp32_logits(fp8_fp4_mqa_logits(...), row_ends)
+ logits = fp8_fp4_mqa_logits(...)
+ use_deep_select = (topk_backend == "deep_select" and logits.dtype == torch.bfloat16
+                    and dcp_world_size == 1 and is_deep_select_supported(logits, topk_tokens))
+ if not use_deep_select:
+     logits = as_fp32_logits(logits, row_ends)
  ...
+ if use_deep_select:
+     deep_select_topk(logits, topk_tokens, end=row_ends, output_idx=topk_indices)
+ else:
+     ops.top_k_per_row_prefill(logits, ..., topk_tokens)
```

Function group: **OP** (a per-call Python function body). Full patch: `patches/62240.patch`.

## 2. Why the obvious hot-patch does nothing

`sparse_attn_indexer` is decorated with `@eager_break_during_capture` (which uses
`functools.wraps`) and registered as a custom op. The op registry keeps the **wrapper** object,
and the wrapper calls the **inner** function through a closure. So replacing the module's
attribute changes nothing the engine actually calls. A toy with the same shape:

In [1]:
# CPU
import functools

def eager_break(fn):                 # same shape as vLLM's decorator
    @functools.wraps(fn)
    def wrapper(*a, **k):
        return fn(*a, **k)           # closes over the INNER function
    return wrapper

@eager_break
def indexer(x):
    return x + 1

registered_op = indexer              # what direct_register_custom_op keeps

# Naive hot-patch: rebind the name to a new version
@eager_break
def indexer(x):
    return x * 100

print("module attribute now returns:", indexer(5))
print("registered op still returns: ", registered_op(5), " <- old code, silent no-op")

module attribute now returns: 500
registered op still returns:  6  <- old code, silent no-op


## 3. How the live lab installs it

`repos/vllm/livelab/fastrsi_lab_ext.py` runs inside every GPU worker. `fastrsi_install`:

1. parses the patched module source and adds any imports the running module lacks
   (#62240 adds `deep_select_topk` and `is_deep_select_supported`);
2. compiles only the target function, without decorators, in the module's own globals, with one
   extra first line that counts calls (the **witness**);
3. follows `__wrapped__` to the innermost function and swaps its `__code__`, so the wrapper, the
   registered op and any recorded graph replays all run the new code;
4. keeps the old code so `fastrsi_uninstall` can put it back.

The same toy, installed the lab's way:

In [2]:
# CPU
import sys, types
sys.path.insert(0, "repos/vllm/livelab")
from fastrsi_lab_ext import LabExtension

BASE = """
import functools
def eager_break(fn):
    @functools.wraps(fn)
    def wrapper(*a, **k):
        return fn(*a, **k)
    return wrapper

@eager_break
def indexer(x):
    return x + 1
"""
PATCHED = BASE.replace("return x + 1", "return x * 100")

mod = types.ModuleType("toy_engine"); mod.__file__ = "toy_engine.py"
exec(BASE, mod.__dict__); sys.modules["toy_engine"] = mod
registered_op = mod.indexer

lab = LabExtension()
print("before:   ", registered_op(5))
lab.fastrsi_install("toy_engine", "indexer", PATCHED)
print("installed:", registered_op(5), " witness:", lab.fastrsi_witness("toy_engine")["hits"])
lab.fastrsi_uninstall("toy_engine", "indexer")
print("undone:   ", registered_op(5))

before:    6
installed: 500  witness: {'toy_engine.indexer': 1}
undone:    6


## 4. Golden tests for the lab (CPU)

Each test is a case a reader can check by eye; the last one pins down the silent no-op from step 2.

In [3]:
# CPU
!python3 repos/vllm/livelab/test_lab_ext.py -v

test_install_changes_what_the_registered_op_runs (__main__.Install.test_install_changes_what_the_registered_op_runs)
The op registry holds the wrapper; after install it must run the new code. ... ok
test_new_import_is_added (__main__.Install.test_new_import_is_added)
The patch imports `sqrt`, which the running module never had. ... ok
test_swapping_the_wrapper_would_be_a_silent_noop (__main__.Install.test_swapping_the_wrapper_would_be_a_silent_noop)
Why the rule exists: rebinding the module attribute leaves the op on old code. ... ok
test_uninstall_restores_base (__main__.Install.test_uninstall_restores_base) ... ok
test_witness_counts_only_new_code (__main__.Install.test_witness_counts_only_new_code) ... ok

----------------------------------------------------------------------
Ran 5 tests in 0.001s

OK


## 5. Start one warm engine (GPU, recorded)

DeepSeek-V4.1-Flash config with dummy weights (same shapes and code paths, meaningless text),
4× GB200, tensor parallel 4. The lab is loaded into every worker with `worker_extension_cls`.
Both arms use `sparse_indexer_topk_backend="deep_select"`: with the default `"auto"`, #62240's
prefill path never turns on.

This start is paid **once**. Everything after it happens on the same engine.

In [4]:
# GPU, recorded
llm = LLM(model=MODEL_DIR, load_format="dummy", tensor_parallel_size=4,
          enable_prefix_caching=True, sparse_indexer_topk_backend="deep_select",
          worker_extension_cls="fastrsi_lab_ext.LabExtension", seed=0)

engine ready in 885.0 s


## 6. Measure the base (GPU, recorded)

A 16K-token prefix is cached first; each measurement then prefills 8K new tokens on top of it
and times one step. Warmups first, then 5 base runs.

In [5]:
# GPU, recorded
def prefill(arm):           # 16K cached prefix + 8K new tokens, one output token
    t = time.time(); llm.generate([TokensPrompt(prompt_token_ids=prefix + fresh_tokens(8192))], sp)
    return time.time() - t

base = [prefill("base") for _ in range(5)]

  warmup     #0: 0.3401 s  cached=16384  24084 new tok/s
  warmup     #1: 0.3284 s  cached=16384  24943 new tok/s
  base       #0: 0.3260 s  cached=16384  25125 new tok/s
  base       #1: 0.3278 s  cached=16384  24987 new tok/s
  base       #2: 0.3317 s  cached=16384  24695 new tok/s
  base       #3: 0.3248 s  cached=16384  25219 new tok/s
  base       #4: 0.3259 s  cached=16384  25135 new tok/s


## 7. Install #62240 on all 4 ranks, live (GPU, recorded)

`collective_rpc` calls the same method on every worker in the same step. Each rank reports the code fingerprint before and after.

In [6]:
# GPU, recorded
src = open("sparse_attn_indexer.py at the #62240 merge").read()
llm.collective_rpc("fastrsi_install", args=(MOD, "sparse_attn_indexer", src))

install took 12.1 ms
  pid 21326: base_hash=6ef40c4dab94, new_hash=c378295907d7
  pid 21328: base_hash=f4b1ed4c224e, new_hash=f760d61af0e0
  pid 21333: base_hash=4628270b2325, new_hash=3d4d2baf0e02
  pid 21339: base_hash=96d087efbbc9, new_hash=73eb15c79fb1


## 8. Prove the new code ran, and measure it (GPU, recorded)

In [7]:
# GPU, recorded
print("witness before:", calls_per_rank())
edited = [prefill("edited") for _ in range(5)]
print("witness after: ", calls_per_rank())

witness before: [0, 0, 0, 0]
  edited     #0: 0.3243 s  cached=16384  25262 new tok/s
  edited     #1: 0.3222 s  cached=16384  25428 new tok/s
  edited     #2: 0.3203 s  cached=16384  25579 new tok/s
  edited     #3: 0.3217 s  cached=16384  25466 new tok/s
  edited     #4: 0.3200 s  cached=16384  25598 new tok/s
witness after:  [40, 40, 40, 40]   <- every rank ran the new code


## 9. Uninstall and check the base comes back (GPU, recorded)

In [8]:
# GPU, recorded
llm.collective_rpc("fastrsi_uninstall", args=(MOD, "sparse_attn_indexer"))
base_again = [prefill("base_again") for _ in range(5)]

uninstall took 0.5 ms
  pid ?: now_hash=6ef40c4dab94
  pid ?: now_hash=f4b1ed4c224e
  pid ?: now_hash=4628270b2325
  pid ?: now_hash=96d087efbbc9
  (each rank is back to its base_hash from step 7)
  base_again #0: 0.3284 s  cached=16384  24947 new tok/s
  base_again #1: 0.3269 s  cached=16384  25063 new tok/s
  base_again #2: 0.3260 s  cached=16384  25132 new tok/s
  base_again #3: 0.3274 s  cached=16384  25023 new tok/s
  base_again #4: 0.3259 s  cached=16384  25138 new tok/s


## 10. What this shows, and what it does not

| | Value |
|---|---:|
| Engine start, paid once | 885 s |
| Install on 4 ranks | 12 ms |
| Uninstall | 0.5 ms |
| Prefill median: base / edited / base again | 0.326 / 0.3217 / 0.3269 s |
| Edited vs base | 1.4% faster |

**Shows:** the change installs on every rank in milliseconds, the witness proves the new code
ran, and uninstall returns every rank to its base code with base timings unchanged.

**Does not show:** real weights; the blog's shape (128K cached + 32K new, where #62240's gain is
largest; 1.4% at this small shape is not comparable to the blog's +15.6%); output equality
(each prefill used fresh random tokens); a signed receipt (the receipt writer is not built yet).